# Notebook 11 — Ablation and Component Sensitivity Study

This notebook is the final component-analysis notebook of the Stage 1 pipeline.
It does **not** replace the final comparison in Notebook 09 and it is **not** a hyperparameter search.
Its role is to check which methodological components affect the results most.

The notebook uses the current frozen outputs from:

- Notebook 02 — preprocessing tensors
- Notebook 03 — GRU baselines
- Notebook 04 — LSTM baselines
- Notebook 05 — Transformer baselines
- Notebook 09 — final global comparison

Main ablations/components checked:

1. Federated optimizer family: FedAvg vs FedProx.
2. Neural architecture: GRU vs LSTM vs Transformer.
3. Learning strategy: Local vs Centralized vs Federated.
4. Modal-density sensitivity: low vs high modal-value proxy density.
5. Class weighting: XGBoost with class weighting ON vs OFF.

Important framing: only the class-weighting ablation retrains a model. The other analyses reuse already saved results.
This keeps the notebook aligned with the current pipeline and avoids adding random experiments after the final comparison.


In [17]:
# 0.1 — Setup
import os, json, warnings, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

try:
    from xgboost import XGBClassifier
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"], check=True)
    from xgboost import XGBClassifier

ACTIVE_CLIENTS = ["England", "Ireland", "USA"]
FEATURE_COLS = ["NH3", "BOD", "DO", "PO4", "pH", "Temp", "Nitrogen", "Nitrate"]
WINDOW = 12
N_FEATS = 8
SEED = 42
PRIMARY = "Macro_F1_present"

OUT_DIR = Path("/kaggle/working/ablation_study")
# Reset output folder to avoid mixing old and new N11 outputs.
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
FIG_DIR = OUT_DIR / "figures"
TAB_DIR = OUT_DIR / "tables"
for d in [OUT_DIR, FIG_DIR, TAB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUT_DIR)
print("Available /kaggle/input folders:")
for p in sorted(Path("/kaggle/input").glob("*")):
    print(" -", p.name)


Output directory: /kaggle/working/ablation_study
Available /kaggle/input folders:
 - datasets


In [18]:
# 0.2 — Robust file discovery helpers
import zipfile

EXTRACT_DIR = Path("/kaggle/working/n11_extracted_inputs")
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# If outputs are attached as zip files, extract them to a temporary working folder.
# If Kaggle already extracted the datasets, this step is harmless.
zip_files = list(Path("/kaggle/input").rglob("*.zip"))
for z in zip_files:
    target = EXTRACT_DIR / z.stem
    target.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(z, "r") as zz:
            zz.extractall(target)
        print(f"✓ extracted zip: {z.name} -> {target}")
    except Exception as e:
        print(f"⚠ could not extract {z}: {e}")

SEARCH_ROOTS = [Path("/kaggle/input"), EXTRACT_DIR]


def find_file(marker):
    for start in SEARCH_ROOTS:
        if not start.exists():
            continue
        for root, _, files in os.walk(start):
            if marker in files:
                return Path(root) / marker
    return None


def find_dir(marker):
    p = find_file(marker)
    return p.parent if p is not None else None


def require_file(marker, explanation=""):
    p = find_file(marker)
    assert p is not None, f"Missing required file: {marker}. {explanation}"
    print(f"✓ {marker}: {p}")
    return p

UNIFIED_PATH = require_file("unified_results_long.csv", "Attach current final_comparison output from N09.")
PREPROCESS_DIR = find_dir("preprocess_manifest.json")
assert PREPROCESS_DIR is not None, "Missing preprocess_manifest.json. Attach current preprocess_outputs from N02."
TENSOR_DIR = PREPROCESS_DIR / "tensors"
assert TENSOR_DIR.exists(), f"Tensor directory missing: {TENSOR_DIR}"
print("✓ preprocess dir:", PREPROCESS_DIR)
print("✓ tensor dir:", TENSOR_DIR)


✓ unified_results_long.csv: /kaggle/input/datasets/hebacheblal/final-comparison/tables/unified_results_long.csv
✓ preprocess dir: /kaggle/input/datasets/hebacheblal/preprocess-outputs
✓ tensor dir: /kaggle/input/datasets/hebacheblal/preprocess-outputs/tensors


## 1 — Model selection before ablation

Before running any ablation, we identify: the best model per client, the best model per training strategy, and the representative models/components used in this notebook. This avoids the unrealistic requirement of ablating every trained model.

In [19]:
# 1.1 — Load final comparison and standardize columns
uni = pd.read_csv(UNIFIED_PATH)
print("Columns in unified table:", list(uni.columns))
needed = ["Family", "Strategy", "Model", "Client", "Macro_F1_present"]
missing = [c for c in needed if c not in uni.columns]
assert not missing, f"unified_results_long.csv missing columns: {missing}"
uni[PRIMARY] = pd.to_numeric(uni[PRIMARY], errors="coerce")
uni = uni.dropna(subset=[PRIMARY]).copy()

best_per_client = (
    uni.sort_values(["Client", PRIMARY], ascending=[True, False])
       .groupby("Client", as_index=False)
       .first()[["Client", "Family", "Strategy", "Model", PRIMARY]]
)

strategy_scores = (
    uni.groupby(["Family", "Strategy", "Model"], as_index=False)[PRIMARY]
       .mean()
       .rename(columns={PRIMARY: "Mean_macro_F1_present"})
       .sort_values("Mean_macro_F1_present", ascending=False)
)
best_per_strategy = (
    strategy_scores.sort_values(["Strategy", "Mean_macro_F1_present"], ascending=[True, False])
                   .groupby("Strategy", as_index=False)
                   .first()
)

print("Best model per client:")
display(best_per_client.round(4))
print("Best representative per strategy:")
display(best_per_strategy.round(4))
best_per_client.round(4).to_csv(TAB_DIR / "best_model_per_client_before_ablation.csv", index=False)
best_per_strategy.round(4).to_csv(TAB_DIR / "best_model_per_strategy_before_ablation.csv", index=False)


Columns in unified table: ['Family', 'Strategy', 'Model', 'Client', 'Accuracy', 'Balanced_accuracy', 'Macro_F1_present', 'Weighted_F1', 'Mean_ordinal_class_error']
Best model per client:


,Client,Family,Strategy,Model,Macro_F1_present
0,England,Classical ML,Local,XGBoost,0.7145
1,Ireland,Classical ML,Centralized,XGBoost,0.5381
2,USA,Deep learning,Local,GRU,0.7909


Best representative per strategy:


,Strategy,Family,Model,Mean_macro_F1_present
0,Centralized,Classical ML,XGBoost,0.6605
1,Federated,Federated learning,LSTM (FedProx_v1),0.4326
2,Local,Classical ML,XGBoost,0.6676


In [20]:
# 1.2 — Model-selection table used for thesis writing
selection_rows = []
for _, r in best_per_client.iterrows():
    selection_rows.append({
        "Selection_reason": f"Best model for {r['Client']}",
        "Client_or_scope": r["Client"],
        "Family": r["Family"],
        "Strategy": r["Strategy"],
        "Model": r["Model"],
        "Macro_F1_present": r[PRIMARY],
        "Used_in_N11": "Reference / model-selection evidence"
    })

fed = uni[uni["Strategy"].str.lower().eq("federated")].copy()
if not fed.empty:
    fed_mean = (fed.groupby(["Family", "Strategy", "Model"], as_index=False)[PRIMARY]
                  .mean()
                  .rename(columns={PRIMARY: "Mean_macro_F1_present"})
                  .sort_values("Mean_macro_F1_present", ascending=False))
    best_fed = fed_mean.iloc[0]
    selection_rows.append({
        "Selection_reason": "Best average federated representative",
        "Client_or_scope": "All clients",
        "Family": best_fed["Family"],
        "Strategy": best_fed["Strategy"],
        "Model": best_fed["Model"],
        "Macro_F1_present": best_fed["Mean_macro_F1_present"],
        "Used_in_N11": "Federated strategy ablation evidence"
    })

selection_df = pd.DataFrame(selection_rows)
print("Models/components selected or referenced in N11:")
display(selection_df.round(4))
selection_df.round(4).to_csv(TAB_DIR / "n11_model_selection_table.csv", index=False)


Models/components selected or referenced in N11:


,Selection_reason,Client_or_scope,Family,Strategy,Model,Macro_F1_present,Used_in_N11
0,Best model for England,England,Classical ML,Local,XGBoost,0.7145,Reference / model-selection evidence
1,Best model for Ireland,Ireland,Classical ML,Centralized,XGBoost,0.5381,Reference / model-selection evidence
2,Best model for USA,USA,Deep learning,Local,GRU,0.7909,Reference / model-selection evidence
3,Best average federated representative,All clients,Federated learning,Federated,LSTM (FedProx_v1),0.4326,Federated strategy ablation evidence


## 2 — Ablation 1: Federated optimizer family (FedAvg vs FedProx)

This section compares FedAvg and FedProx results already obtained in the federated experiments.

Two views are reported:

1. **Best-family comparison:** the best available FedAvg result is compared with the best available FedProx result. This is useful for thesis-level positioning, but it is **not** a pure same-architecture optimizer ablation if the best FedAvg and best FedProx rows come from different architectures.
2. **Matched-architecture comparison, when available:** FedAvg and FedProx are compared inside the same architecture family (GRU, LSTM, Transformer). This is the cleaner algorithm-level comparison.

Thesis-safe interpretation: if FedProx is slightly higher than FedAvg, this should be presented as a limited improvement inside a still-weak federated regime, not as evidence that FedProx solves the non-IID problem.


In [21]:
# 2.1 — FedAvg vs FedProx from final comparison
fed = uni[uni["Strategy"].str.lower().eq("federated")].copy()
assert not fed.empty, "No federated rows found in unified_results_long.csv."

fed["Federated_algorithm"] = np.where(
    fed["Model"].str.contains("FedProx", case=False, na=False),
    "FedProx",
    np.where(fed["Model"].str.contains("FedAvg", case=False, na=False), "FedAvg", "Other")
)

def infer_architecture(model_name):
    name = str(model_name).lower()
    if "transformer" in name:
        return "Transformer"
    if "lstm" in name:
        return "LSTM"
    if "gru" in name:
        return "GRU"
    return "Unknown"

fed["Architecture"] = fed["Model"].apply(infer_architecture)

# View A: best-family comparison (not necessarily same architecture)
fed_alg = (
    fed[fed["Federated_algorithm"].isin(["FedAvg", "FedProx"])]
    .groupby(["Federated_algorithm", "Model", "Architecture"], as_index=False)[PRIMARY]
    .mean()
    .rename(columns={PRIMARY: "Mean_macro_F1_present"})
    .sort_values("Mean_macro_F1_present", ascending=False)
)

best_alg = (
    fed_alg.sort_values(["Federated_algorithm", "Mean_macro_F1_present"], ascending=[True, False])
    .groupby("Federated_algorithm", as_index=False)
    .first()
)

print("Ablation 1A — best FedAvg family result vs best FedProx family result:")
display(best_alg.round(4))
best_alg.round(4).to_csv(TAB_DIR / "ablation_1_fedavg_vs_fedprox.csv", index=False)
best_alg.round(4).to_csv(TAB_DIR / "ablation_1a_fedavg_vs_fedprox_best_family.csv", index=False)

if set(best_alg["Federated_algorithm"]) >= {"FedAvg", "FedProx"}:
    fa = float(best_alg.loc[best_alg["Federated_algorithm"] == "FedAvg", "Mean_macro_F1_present"].iloc[0])
    fp = float(best_alg.loc[best_alg["Federated_algorithm"] == "FedProx", "Mean_macro_F1_present"].iloc[0])
    print(f"Best-family FedProx - FedAvg = {fp-fa:+.4f}")
else:
    fa = fp = np.nan
    print("Only one federated algorithm family found; best-family delta not computed.")

# View B: matched architecture comparison when both algorithms exist for the same architecture
matched_fed = (
    fed_alg[fed_alg["Architecture"].ne("Unknown")]
    .groupby(["Architecture", "Federated_algorithm"], as_index=False)["Mean_macro_F1_present"]
    .max()
)
matched_pivot = matched_fed.pivot(index="Architecture", columns="Federated_algorithm", values="Mean_macro_F1_present").reset_index()
if {"FedAvg", "FedProx"}.issubset(matched_pivot.columns):
    matched_pivot["Delta_FedProx_minus_FedAvg"] = matched_pivot["FedProx"] - matched_pivot["FedAvg"]
    print("Ablation 1B — matched architecture FedAvg vs FedProx comparison:")
    display(matched_pivot.round(4))
    matched_pivot.round(4).to_csv(TAB_DIR / "ablation_1b_fedavg_vs_fedprox_matched_architecture.csv", index=False)
else:
    print("Matched architecture FedAvg/FedProx comparison is not available from the final comparison file.")
    pd.DataFrame(columns=["Architecture", "FedAvg", "FedProx", "Delta_FedProx_minus_FedAvg"]).to_csv(
        TAB_DIR / "ablation_1b_fedavg_vs_fedprox_matched_architecture.csv", index=False
    )


Ablation 1A — best FedAvg family result vs best FedProx family result:


,Federated_algorithm,Model,Architecture,Mean_macro_F1_present
0,FedAvg,LSTM (FedAvg_v3),LSTM,0.4240
1,FedProx,LSTM (FedProx_v1),LSTM,0.4326


Best-family FedProx - FedAvg = +0.0087
Ablation 1B — matched architecture FedAvg vs FedProx comparison:


Federated_algorithm,Architecture,FedAvg,FedProx,Delta_FedProx_minus_FedAvg
0,GRU,0.4183,0.3934,-0.0249
1,LSTM,0.4240,0.4326,0.0087
2,Transformer,0.2387,0.3509,0.1123


## 3 — Ablation 2: GRU vs LSTM vs Transformer

This ablation compares model architecture while keeping the task, input tensors, clients, and evaluation metric consistent.

In [22]:
# 3.1 — Architecture ablation from unified final comparison
dl = uni[uni["Family"].str.contains("Deep", case=False, na=False)].copy()

def arch_name(model):
    m = str(model).lower()
    if "transformer" in m:
        return "Transformer"
    if "lstm" in m:
        return "LSTM"
    if "gru" in m:
        return "GRU"
    return str(model)

dl["Architecture"] = dl["Model"].map(arch_name)
arch_strategy = (dl.groupby(["Strategy", "Architecture"], as_index=False)[PRIMARY]
                   .mean()
                   .rename(columns={PRIMARY: "Mean_macro_F1_present"})
                   .sort_values(["Strategy", "Mean_macro_F1_present"], ascending=[True, False]))
print("Ablation 2 — architecture comparison by strategy:")
display(arch_strategy.round(4))
arch_strategy.round(4).to_csv(TAB_DIR / "ablation_2_architecture_comparison.csv", index=False)
pivot_arch = arch_strategy.pivot(index="Architecture", columns="Strategy", values="Mean_macro_F1_present")
display(pivot_arch.round(4))


Ablation 2 — architecture comparison by strategy:


,Strategy,Architecture,Mean_macro_F1_present
0,Centralized,GRU,0.6586
2,Centralized,Transformer,0.6578
1,Centralized,LSTM,0.6455
5,Local,Transformer,0.6647
3,Local,GRU,0.6571
4,Local,LSTM,0.6460


Strategy,Centralized,Local
Architecture,,
GRU,0.6586,0.6571
LSTM,0.6455,0.6460
Transformer,0.6578,0.6647


## 4 — Ablation 3: Learning strategy sensitivity

This summarizes Local vs Centralized vs Federated performance. It is a high-level ablation of the training strategy.

In [23]:
# 4.1 — Learning-strategy sensitivity
# Reviewer-safe version:
#   View A = best representative per strategy (primary thesis table; aligns with N09).
#   View B = average over all configurations (supplementary robustness view).
#
# Why both?
# After corrected N09, the federated strategy contains many FL variants.
# Averaging all of them is useful, but it should not be confused with the
# best federated representative used in the final model comparison.

strategy_mean_all_configs = (
    uni.groupby("Strategy", as_index=False)[PRIMARY]
       .mean()
       .rename(columns={PRIMARY: "Mean_macro_F1_present"})
       .sort_values("Mean_macro_F1_present", ascending=False)
)

strategy_best_representative = (
    strategy_scores.sort_values(["Strategy", "Mean_macro_F1_present"], ascending=[True, False])
                   .groupby("Strategy", as_index=False)
                   .first()
                   .sort_values("Mean_macro_F1_present", ascending=False)
)

# This variable is kept for backward compatibility with the synthesis/figure cells.
# It now refers to the primary thesis-safe view: best representative per strategy.
strategy_mean = strategy_best_representative[["Strategy", "Mean_macro_F1_present"]].copy()

print("Ablation 3A — learning strategy: best representative per strategy")
display(strategy_best_representative.round(4))

print("Ablation 3B — learning strategy: average over all configurations")
display(strategy_mean_all_configs.round(4))

strategy_best_representative.round(4).to_csv(
    TAB_DIR / "ablation_3_learning_strategy_best_representative.csv",
    index=False
)
strategy_mean_all_configs.round(4).to_csv(
    TAB_DIR / "ablation_3_learning_strategy_all_configurations.csv",
    index=False
)

# Keep the original filename as the primary thesis-safe table.
strategy_mean.round(4).to_csv(TAB_DIR / "ablation_3_learning_strategy.csv", index=False)


Ablation 3A — learning strategy: best representative per strategy


,Strategy,Family,Model,Mean_macro_F1_present
2,Local,Classical ML,XGBoost,0.6676
0,Centralized,Classical ML,XGBoost,0.6605
1,Federated,Federated learning,LSTM (FedProx_v1),0.4326


Ablation 3B — learning strategy: average over all configurations


,Strategy,Mean_macro_F1_present
2,Local,0.6504
0,Centralized,0.6165
1,Federated,0.3035


## 5 — Ablation 4: Modal-density / imputed-data sensitivity

The modal-value proxy was never an input feature. Therefore, we cannot perform a direct "with vs without modal-proxy feature" ablation.

Instead, we use already-computed modal-density stratified evaluation: performance on low-modal-density test sequences versus high-modal-density test sequences. This addresses the supervisor's question about the influence of imputed/modal-value-dominated data.

Important: this is a diagnostic proxy, not proof that every modal-value cell was imputed.

In [24]:
# 5.1 — Load available modal-density stratified evaluations
modal_files = []
for marker in ["modal_density_stratified_eval.csv", "lstm_modal_density_stratified_eval.csv", "transformer_modal_density_stratified_eval.csv"]:
    p = find_file(marker)
    if p is not None:
        modal_files.append(p)
        print("✓", marker, "->", p)
assert modal_files, "No modal-density stratified evaluation CSV found. Attach GRU/LSTM/Transformer baseline outputs."

modal_frames = []
for p in modal_files:
    df = pd.read_csv(p)
    name = p.name.lower()
    if "lstm" in name:
        src = "LSTM"
    elif "transformer" in name:
        src = "Transformer"
    else:
        src = "GRU"
    df["Source_notebook_model_family"] = src
    modal_frames.append(df)
modal_df = pd.concat(modal_frames, ignore_index=True)
print("Modal-density columns:", list(modal_df.columns))
metric_candidates = ["Macro_F1_present_classes", "Macro_F1_present"]
metric_col = next((c for c in metric_candidates if c in modal_df.columns), None)
assert metric_col is not None, "Could not find present-class Macro-F1 column in modal-density file."
needed_cols = [c for c in ["Source_notebook_model_family", "Model", "Client", "Stratum", metric_col] if c in modal_df.columns]
display(modal_df[needed_cols].head())

index_cols = ["Source_notebook_model_family", "Client"]
if "Model" in modal_df.columns:
    index_cols.insert(1, "Model")
modal_pivot = modal_df.pivot_table(index=index_cols, columns="Stratum", values=metric_col, aggfunc="first").reset_index()
if "High_modal_density" in modal_pivot.columns and "Low_modal_density" in modal_pivot.columns:
    modal_pivot["Delta_high_minus_low"] = modal_pivot["High_modal_density"] - modal_pivot["Low_modal_density"]
else:
    print("Available strata:", modal_pivot.columns.tolist())
print("Ablation 4 — modal-density sensitivity:")
display(modal_pivot.round(4))
modal_pivot.round(4).to_csv(TAB_DIR / "ablation_4_modal_density_sensitivity.csv", index=False)


✓ modal_density_stratified_eval.csv -> /kaggle/input/datasets/hebacheblal/gru-baselines/audit/modal_density_stratified_eval.csv
✓ lstm_modal_density_stratified_eval.csv -> /kaggle/input/datasets/hebacheblal/lstm-baselines/audit/lstm_modal_density_stratified_eval.csv
✓ transformer_modal_density_stratified_eval.csv -> /kaggle/input/datasets/hebacheblal/transformer-baselines/audit/transformer_modal_density_stratified_eval.csv
Modal-density columns: ['Model', 'Client', 'Stratum', 'N', 'Accuracy', 'Macro_F1_present_classes', 'Weighted_F1', 'Source_notebook_model_family']


,Source_notebook_model_family,Model,Client,Stratum,Macro_F1_present_classes
0,GRU,Local,England,Low_modal_density,0.5672
1,GRU,Local,England,High_modal_density,0.7277
2,GRU,Local,Ireland,Low_modal_density,0.4707
3,GRU,Local,Ireland,High_modal_density,0.4511
4,GRU,Local,USA,Low_modal_density,0.7963


Ablation 4 — modal-density sensitivity:


Stratum,Source_notebook_model_family,Model,Client,High_modal_density,Low_modal_density,Delta_high_minus_low
0,GRU,Centralized,England,0.7227,0.5722,0.1505
1,GRU,Centralized,Ireland,0.4456,0.5098,-0.0642
2,GRU,Centralized,USA,0.4933,0.7795,-0.2862
3,GRU,FedAvg_v3,England,0.2940,0.2012,0.0928
4,GRU,FedAvg_v3,Ireland,0.3471,0.3708,-0.0237
5,GRU,FedAvg_v3,USA,0.5392,0.5697,-0.0305
6,GRU,Local,England,0.7277,0.5672,0.1605
7,GRU,Local,Ireland,0.4511,0.4707,-0.0196
8,GRU,Local,USA,0.5215,0.7963,-0.2748
9,LSTM,Centralized LSTM,England,0.6987,0.5579,0.1408


## 6 — Ablation 5: Class weighting ON vs OFF using XGBoost

This is the only new retraining experiment in N11. It is run on XGBoost because it is one of the strongest and fastest models, and because it isolates the effect of class weighting without expensive deep-model retraining.

The XGBoost hyperparameters are kept aligned with Notebook 08; the only change is whether `sample_weight` is passed during fitting.

Important interpretation rule:
- Do **not** claim that class weighting improves Macro-F1 on every client.
- Report both present-class Macro-F1 and balanced accuracy.
- If class weighting reduces Macro-F1 for Ireland/USA but improves balanced accuracy, this means the weighting changes the model away from majority-class behavior; it is still scientifically useful for class-imbalance handling.


In [25]:
# 6.1 — Load Stage-1 tensors and flatten 12x8 sequences for XGBoost

def load_client_npz(client):
    p = TENSOR_DIR / f"{client.lower()}_stage1_sequences.npz"
    assert p.exists(), f"Missing tensor file: {p}"
    arr = np.load(p)
    X_train = arr["X_train"].astype(np.float32)
    y_train = arr["y_train"].astype(int)
    X_test = arr["X_test"].astype(np.float32)
    y_test = arr["y_test"].astype(int)
    arr.close()
    return {
        "X_train_seq": X_train,
        "y_train": y_train,
        "X_test_seq": X_test,
        "y_test": y_test,
        "X_train_flat": X_train.reshape(X_train.shape[0], WINDOW*N_FEATS),
        "X_test_flat": X_test.reshape(X_test.shape[0], WINDOW*N_FEATS),
    }

tensor_data = {client: load_client_npz(client) for client in ACTIVE_CLIENTS}
for client, d in tensor_data.items():
    print(client, d["X_train_seq"].shape, d["X_test_seq"].shape)


England (76530, 12, 8) (11942, 12, 8)
Ireland (27778, 12, 8) (5887, 12, 8)
USA (30222, 12, 8) (1979, 12, 8)


In [26]:
# 6.2 — XGBoost train/evaluate function aligned with N08

def present_macro_f1(y_true, y_pred):
    present = np.unique(y_true)
    return f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)


def make_xgb_model():
    return XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multi:softprob",
        tree_method="hist",
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1
    )


def train_eval_xgb(X_train, y_train, X_test, y_test, weighted=True):
    model = make_xgb_model()
    sample_weight = compute_sample_weight(class_weight="balanced", y=y_train) if weighted else None
    model.fit(X_train, y_train, sample_weight=sample_weight)
    pred = model.predict(X_test)
    return {
        "Accuracy": accuracy_score(y_test, pred),
        "Balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "Macro_F1_present": present_macro_f1(y_test, pred),
        "Weighted_F1": f1_score(y_test, pred, average="weighted", zero_division=0),
    }


In [27]:
# 6.3 — Run class weighting ablation
class_weight_rows = []
for client in ACTIVE_CLIENTS:
    d = tensor_data[client]
    metrics_on = train_eval_xgb(d["X_train_flat"], d["y_train"], d["X_test_flat"], d["y_test"], weighted=True)
    metrics_off = train_eval_xgb(d["X_train_flat"], d["y_train"], d["X_test_flat"], d["y_test"], weighted=False)
    row = {
        "Client": client,
        "Macro_F1_weighted_ON": metrics_on["Macro_F1_present"],
        "Macro_F1_weighted_OFF": metrics_off["Macro_F1_present"],
        "Delta_ON_minus_OFF": metrics_on["Macro_F1_present"] - metrics_off["Macro_F1_present"],
        "Accuracy_ON": metrics_on["Accuracy"],
        "Accuracy_OFF": metrics_off["Accuracy"],
        "Balanced_accuracy_ON": metrics_on["Balanced_accuracy"],
        "Balanced_accuracy_OFF": metrics_off["Balanced_accuracy"],
        "Delta_balanced_accuracy_ON_minus_OFF": metrics_on["Balanced_accuracy"] - metrics_off["Balanced_accuracy"],
    }
    row["Macro_F1_direction"] = "ON better" if row["Delta_ON_minus_OFF"] > 0 else "OFF better"
    row["Balanced_accuracy_direction"] = "ON better" if row["Delta_balanced_accuracy_ON_minus_OFF"] > 0 else "OFF better"
    class_weight_rows.append(row)
    print(
        f"{client}: Macro-F1 ON={row['Macro_F1_weighted_ON']:.4f}, "
        f"OFF={row['Macro_F1_weighted_OFF']:.4f}, "
        f"delta={row['Delta_ON_minus_OFF']:+.4f}; "
        f"balanced-acc delta={row['Delta_balanced_accuracy_ON_minus_OFF']:+.4f}"
    )

class_weight_df = pd.DataFrame(class_weight_rows)
print("Ablation 5 — class weighting ON vs OFF:")
display(class_weight_df.round(4))
class_weight_df.round(4).to_csv(TAB_DIR / "ablation_5_xgboost_class_weighting_on_off.csv", index=False)

class_weight_interpretation = pd.DataFrame([
    {
        "Metric": "Present-class Macro-F1",
        "Mean_delta_ON_minus_OFF": class_weight_df["Delta_ON_minus_OFF"].mean(),
        "Clients_improved_by_ON": int((class_weight_df["Delta_ON_minus_OFF"] > 0).sum()),
        "Clients_worse_by_ON": int((class_weight_df["Delta_ON_minus_OFF"] < 0).sum()),
        "Safe_interpretation": "Class weighting improves England Macro-F1 but not Ireland/USA Macro-F1; do not claim universal improvement."
    },
    {
        "Metric": "Balanced accuracy",
        "Mean_delta_ON_minus_OFF": class_weight_df["Delta_balanced_accuracy_ON_minus_OFF"].mean(),
        "Clients_improved_by_ON": int((class_weight_df["Delta_balanced_accuracy_ON_minus_OFF"] > 0).sum()),
        "Clients_worse_by_ON": int((class_weight_df["Delta_balanced_accuracy_ON_minus_OFF"] < 0).sum()),
        "Safe_interpretation": "Class weighting improves balanced accuracy on all clients, supporting its use for class-imbalance handling."
    }
])
print("Class-weighting interpretation summary:")
display(class_weight_interpretation.round(4))
class_weight_interpretation.round(4).to_csv(TAB_DIR / "ablation_5_class_weighting_interpretation.csv", index=False)


England: Macro-F1 ON=0.7145, OFF=0.6538, delta=+0.0607; balanced-acc delta=+0.0773
Ireland: Macro-F1 ON=0.4995, OFF=0.5249, delta=-0.0254; balanced-acc delta=+0.0531
USA: Macro-F1 ON=0.7889, OFF=0.8038, delta=-0.0149; balanced-acc delta=+0.0788
Ablation 5 — class weighting ON vs OFF:


,Client,Macro_F1_weighted_ON,Macro_F1_weighted_OFF,Delta_ON_minus_OFF,Accuracy_ON,Accuracy_OFF,Balanced_accuracy_ON,Balanced_accuracy_OFF,Delta_balanced_accuracy_ON_minus_OFF,Macro_F1_direction,Balanced_accuracy_direction
0,England,0.7145,0.6538,0.0607,0.8180,0.8257,0.7016,0.6243,0.0773,ON better,ON better
1,Ireland,0.4995,0.5249,-0.0254,0.8648,0.9222,0.5550,0.5019,0.0531,OFF better,ON better
2,USA,0.7889,0.8038,-0.0149,0.9096,0.9394,0.8307,0.7518,0.0788,OFF better,ON better


Class-weighting interpretation summary:


,Metric,Mean_delta_ON_minus_OFF,Clients_improved_by_ON,Clients_worse_by_ON,Safe_interpretation
0,Present-class Macro-F1,0.0068,1,2,Class weighting improves England Macro-F1 but ...
1,Balanced accuracy,0.0697,3,0,Class weighting improves balanced accuracy on ...


## 7 — Synthesis table

This table is the final ablation-study output to report in Chapter 4.

In [28]:
# 7.1 — Build final synthesis (scientifically safe + no NaN idxmax error)

summary_rows = []

def add_summary(ablation, source, main_result, interpretation):
    summary_rows.append({
        "Ablation": ablation,
        "Source": source,
        "Main_result": main_result,
        "Interpretation": interpretation,
    })

# ------------------------------------------------------------
# Ablation 1A — FedAvg vs FedProx: best-family comparison
# IMPORTANT: this is NOT necessarily same architecture.
# ------------------------------------------------------------
if "best_alg" in globals() and isinstance(best_alg, pd.DataFrame) and not best_alg.empty:
    tmp = best_alg.copy()
    if {"Federated_algorithm", "Model", "Mean_macro_F1_present"}.issubset(tmp.columns):
        desc = "; ".join([
            f"{r['Federated_algorithm']}: {float(r['Mean_macro_F1_present']):.3f} ({r['Model']})"
            for _, r in tmp.iterrows()
        ])
        add_summary(
            "FedAvg vs FedProx",
            "Existing federated results",
            desc,
            "Best-family comparison only; not a strict same-architecture optimizer ablation. "
            "It is used to summarize which federated family gave the strongest representative result. "
            "Both federated families remain weaker than the strongest non-federated baselines."
        )
    else:
        add_summary(
            "FedAvg vs FedProx",
            "Existing federated results",
            "Not available: expected columns missing in best_alg.",
            "The comparison was skipped because the required columns were not available."
        )
else:
    add_summary(
        "FedAvg vs FedProx",
        "Existing federated results",
        "Not available",
        "No best_alg table was available in the notebook state."
    )

# ------------------------------------------------------------
# Ablation 1B — FedAvg vs FedProx: matched architecture
# Only report this when at least one valid same-architecture delta exists.
# This avoids the KeyError: nan caused by idxmax() on all-NaN values.
# ------------------------------------------------------------
if "matched_pivot" in globals() and isinstance(matched_pivot, pd.DataFrame):
    mp = matched_pivot.copy()
    required = {"Architecture", "FedAvg", "FedProx"}

    if (not mp.empty) and required.issubset(mp.columns):
        if "Delta_FedProx_minus_FedAvg" not in mp.columns:
            mp["Delta_FedProx_minus_FedAvg"] = mp["FedProx"] - mp["FedAvg"]

        mp["Delta_FedProx_minus_FedAvg"] = pd.to_numeric(
            mp["Delta_FedProx_minus_FedAvg"], errors="coerce"
        )
        valid_mp = mp.dropna(subset=["Delta_FedProx_minus_FedAvg"])

        if not valid_mp.empty:
            idx = valid_mp["Delta_FedProx_minus_FedAvg"].abs().idxmax()
            best_matched = valid_mp.loc[idx]
            add_summary(
                "FedAvg vs FedProx, matched architecture",
                "Existing federated results parsed by architecture",
                f"Largest matched delta: {best_matched['Architecture']} = {best_matched['Delta_FedProx_minus_FedAvg']:+.3f}",
                "This is the cleaner optimizer-level view because FedAvg and FedProx are compared within the same architecture."
            )
        else:
            add_summary(
                "FedAvg vs FedProx, matched architecture",
                "Existing federated results parsed by architecture",
                "Not available: no architecture has both valid FedAvg and valid FedProx scores.",
                "No strict same-architecture optimizer ablation is reported. This avoids claiming a matched comparison that the available table does not support."
            )
    else:
        add_summary(
            "FedAvg vs FedProx, matched architecture",
            "Existing federated results parsed by architecture",
            "Not available",
            "The matched table is empty or missing Architecture/FedAvg/FedProx columns."
        )

# ------------------------------------------------------------
# Ablation 2 — GRU vs LSTM vs Transformer
# ------------------------------------------------------------
if "arch_strategy" in globals() and isinstance(arch_strategy, pd.DataFrame) and not arch_strategy.empty:
    ar = arch_strategy.copy()
    if {"Strategy", "Architecture", "Mean_macro_F1_present"}.issubset(ar.columns):
        local_arch = ar[ar["Strategy"].astype(str).str.lower().eq("local")]
        use_arch = local_arch if not local_arch.empty else ar
        best = use_arch.sort_values("Mean_macro_F1_present", ascending=False).iloc[0]

        if not use_arch.empty:
            spread = use_arch["Mean_macro_F1_present"].max() - use_arch["Mean_macro_F1_present"].min()
        else:
            spread = np.nan

        add_summary(
            "GRU vs LSTM vs Transformer",
            "Existing architecture results",
            f"Best local architecture: {best['Architecture']} ({float(best['Mean_macro_F1_present']):.3f}); spread = {float(spread):.3f}",
            "Architecture differences are small; no single deep architecture dominates all clients. "
            "This supports the interpretation that data heterogeneity and class imbalance are stronger bottlenecks than architecture choice."
        )
    else:
        add_summary(
            "GRU vs LSTM vs Transformer",
            "Existing architecture results",
            "Not available: expected columns missing.",
            "Architecture comparison was skipped because the required columns were not available."
        )
else:
    add_summary(
        "GRU vs LSTM vs Transformer",
        "Existing architecture results",
        "Not available",
        "Architecture comparison table was not available in the notebook state."
    )

# ------------------------------------------------------------
# Ablation 3 — Local vs Centralized vs Federated
# ------------------------------------------------------------
if "strategy_best_representative" in globals() and isinstance(strategy_best_representative, pd.DataFrame) and not strategy_best_representative.empty:
    sb = strategy_best_representative.copy()
    if {"Strategy", "Model", "Mean_macro_F1_present"}.issubset(sb.columns):
        best = sb.sort_values("Mean_macro_F1_present", ascending=False).iloc[0]

        rep_parts = []
        for _, r in sb.sort_values("Strategy").iterrows():
            rep_parts.append(
                f"{r['Strategy']}: {float(r['Mean_macro_F1_present']):.3f} ({r['Model']})"
            )
        rep_text = "; ".join(rep_parts)

        if "strategy_mean_all_configs" in globals() and isinstance(strategy_mean_all_configs, pd.DataFrame):
            all_parts = []
            for _, r in strategy_mean_all_configs.sort_values("Strategy").iterrows():
                all_parts.append(
                    f"{r['Strategy']}: {float(r['Mean_macro_F1_present']):.3f}"
                )
            all_text = "; ".join(all_parts)
        else:
            all_text = "not available"

        add_summary(
            "Local vs Centralized vs Federated",
            "Final comparison results",
            f"Best-representative view: {rep_text}. All-configuration average: {all_text}.",
            "The best-representative view is the primary thesis-safe comparison because it aligns with N09 model selection. "
            "The all-configuration average is kept as a supplementary robustness view and should not be confused with the final best-model ranking."
        )
    else:
        add_summary(
            "Local vs Centralized vs Federated",
            "Final comparison results",
            "Not available: expected columns missing.",
            "Strategy comparison was skipped because the required columns were not available."
        )
else:
    add_summary(
        "Local vs Centralized vs Federated",
        "Final comparison results",
        "Not available",
        "Learning-strategy comparison table was not available in the notebook state."
    )

# ------------------------------------------------------------
# Ablation 4 — Modal-density sensitivity
# NOTE: this is a stratified sensitivity analysis, not a direct ON/OFF input ablation.
# ------------------------------------------------------------
if "modal_pivot" in globals() and isinstance(modal_pivot, pd.DataFrame) and not modal_pivot.empty:
    md = modal_pivot.copy()
    if "Delta_high_minus_low" in md.columns:
        md["Delta_high_minus_low"] = pd.to_numeric(md["Delta_high_minus_low"], errors="coerce")
        valid_md = md.dropna(subset=["Delta_high_minus_low"])

        if not valid_md.empty:
            idx = valid_md["Delta_high_minus_low"].abs().idxmax()
            max_abs = valid_md.loc[idx]
            model_text = max_abs.get("Model", max_abs.get("Source_notebook_model_family", "model"))

            add_summary(
                "Low vs high modal-density sequences",
                "Existing modal-density stratified evaluation",
                f"Largest absolute delta: {max_abs['Client']} / {model_text} = {max_abs['Delta_high_minus_low']:+.3f}",
                "Modal-value dominance changes measured performance. This is especially important for interpreting USA, "
                "where high modal-density sequences show weaker class-balanced behavior."
            )
        else:
            add_summary(
                "Low vs high modal-density sequences",
                "Existing modal-density stratified evaluation",
                "Not available: all modal-density deltas are missing.",
                "The modal-density table exists, but no finite high-minus-low delta could be computed."
            )
    else:
        add_summary(
            "Low vs high modal-density sequences",
            "Existing modal-density stratified evaluation",
            "Not available: Delta_high_minus_low column missing.",
            "Modal-density sensitivity could not be summarized automatically."
        )
else:
    add_summary(
        "Low vs high modal-density sequences",
        "Existing modal-density stratified evaluation",
        "Not available",
        "Modal-density sensitivity table was not available in the notebook state."
    )

# ------------------------------------------------------------
# Ablation 5 — Class weighting ON vs OFF
# This is the cleanest true retraining ablation in N11.
# ------------------------------------------------------------
if "class_weight_df" in globals() and isinstance(class_weight_df, pd.DataFrame) and not class_weight_df.empty:
    cw = class_weight_df.copy()
    needed = {"Delta_ON_minus_OFF", "Delta_balanced_accuracy_ON_minus_OFF"}
    if needed.issubset(cw.columns):
        cw["Delta_ON_minus_OFF"] = pd.to_numeric(cw["Delta_ON_minus_OFF"], errors="coerce")
        cw["Delta_balanced_accuracy_ON_minus_OFF"] = pd.to_numeric(
            cw["Delta_balanced_accuracy_ON_minus_OFF"], errors="coerce"
        )

        mean_macro_delta = cw["Delta_ON_minus_OFF"].mean()
        mean_bal_delta = cw["Delta_balanced_accuracy_ON_minus_OFF"].mean()
        improved_macro = int((cw["Delta_ON_minus_OFF"] > 0).sum())
        improved_bal = int((cw["Delta_balanced_accuracy_ON_minus_OFF"] > 0).sum())

        add_summary(
            "Class weighting ON vs OFF",
            "New XGBoost retraining ablation",
            f"Mean Macro-F1 delta = {mean_macro_delta:+.3f} ({improved_macro}/3 clients improved); "
            f"mean balanced-accuracy delta = {mean_bal_delta:+.3f} ({improved_bal}/3 clients improved)",
            "Class weighting has mixed effects on present-class Macro-F1, but it improves balanced accuracy across clients. "
            "It is therefore justified as an imbalance-aware choice, not as a universal Macro-F1 improvement."
        )
    else:
        add_summary(
            "Class weighting ON vs OFF",
            "New XGBoost retraining ablation",
            "Not available: expected delta columns missing.",
            "Class-weighting ablation was run, but the summary columns could not be found."
        )
else:
    add_summary(
        "Class weighting ON vs OFF",
        "New XGBoost retraining ablation",
        "Not available",
        "Class-weighting ablation table was not available in the notebook state."
    )

# ------------------------------------------------------------
# Methodological notes: requested components that should not be overclaimed
# ------------------------------------------------------------
add_summary(
    "With vs without modal-value proxy as input",
    "Methodological check",
    "Not applicable: modal proxy was never used as a model input feature.",
    "The correct analysis is modal-density stratified evaluation, not dropping a non-existing input."
)

add_summary(
    "Full preprocessing effect",
    "Future work / requires N02 rerun",
    "Not run in N11 to avoid invalid approximations.",
    "A valid preprocessing ablation requires rebuilding tensors from raw/monthly data. "
    "The notebook avoids using an unsafe proxy for this component."
)

# ------------------------------------------------------------
# Save final synthesis
# ------------------------------------------------------------
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

summary_df.to_csv(TAB_DIR / "n11_ablation_synthesis.csv", index=False)
summary_df.to_csv(TAB_DIR / "n11_ablation_synthesis_fixed.csv", index=False)

print("✓ saved:", TAB_DIR / "n11_ablation_synthesis.csv")
print("✓ saved:", TAB_DIR / "n11_ablation_synthesis_fixed.csv")


,Ablation,Source,Main_result,Interpretation
0,FedAvg vs FedProx,Existing federated results,FedAvg: 0.424 (LSTM (FedAvg_v3)); FedProx: 0.4...,Best-family comparison only; not a strict same...
1,"FedAvg vs FedProx, matched architecture",Existing federated results parsed by architecture,Largest matched delta: Transformer = +0.112,This is the cleaner optimizer-level view becau...
2,GRU vs LSTM vs Transformer,Existing architecture results,Best local architecture: Transformer (0.665); ...,Architecture differences are small; no single ...
3,Local vs Centralized vs Federated,Final comparison results,Best-representative view: Centralized: 0.660 (...,The best-representative view is the primary th...
4,Low vs high modal-density sequences,Existing modal-density stratified evaluation,Largest absolute delta: USA / Local Transforme...,Modal-value dominance changes measured perform...
5,Class weighting ON vs OFF,New XGBoost retraining ablation,Mean Macro-F1 delta = +0.007 (1/3 clients impr...,Class weighting has mixed effects on present-c...
6,With vs without modal-value proxy as input,Methodological check,Not applicable: modal proxy was never used as ...,The correct analysis is modal-density stratifi...
7,Full preprocessing effect,Future work / requires N02 rerun,Not run in N11 to avoid invalid approximations.,A valid preprocessing ablation requires rebuil...


✓ saved: /kaggle/working/ablation_study/tables/n11_ablation_synthesis.csv
✓ saved: /kaggle/working/ablation_study/tables/n11_ablation_synthesis_fixed.csv


In [29]:
# 7.2 — Figures
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(class_weight_df)); w = 0.36
ax.bar(x - w/2, class_weight_df["Macro_F1_weighted_ON"], width=w, label="Class weighting ON")
ax.bar(x + w/2, class_weight_df["Macro_F1_weighted_OFF"], width=w, label="Class weighting OFF")
ax.set_xticks(x); ax.set_xticklabels(class_weight_df["Client"])
ax.set_ylim(0, 1); ax.set_ylabel("Present-class Macro-F1")
ax.set_title("Class weighting ablation — XGBoost")
ax.legend(); ax.grid(axis="y", alpha=0.3)
for i, row in class_weight_df.iterrows():
    ax.text(i-w/2, row["Macro_F1_weighted_ON"] + 0.01, f"{row['Macro_F1_weighted_ON']:.3f}", ha="center", fontsize=8)
    ax.text(i+w/2, row["Macro_F1_weighted_OFF"] + 0.01, f"{row['Macro_F1_weighted_OFF']:.3f}", ha="center", fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "ablation_class_weighting_xgboost.png", dpi=180, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, class_weight_df["Balanced_accuracy_ON"], width=w, label="Class weighting ON")
ax.bar(x + w/2, class_weight_df["Balanced_accuracy_OFF"], width=w, label="Class weighting OFF")
ax.set_xticks(x); ax.set_xticklabels(class_weight_df["Client"])
ax.set_ylim(0, 1); ax.set_ylabel("Balanced accuracy")
ax.set_title("Class weighting ablation — balanced accuracy")
ax.legend(); ax.grid(axis="y", alpha=0.3)
for i, row in class_weight_df.iterrows():
    ax.text(i-w/2, row["Balanced_accuracy_ON"] + 0.01, f"{row['Balanced_accuracy_ON']:.3f}", ha="center", fontsize=8)
    ax.text(i+w/2, row["Balanced_accuracy_OFF"] + 0.01, f"{row['Balanced_accuracy_OFF']:.3f}", ha="center", fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "ablation_class_weighting_balanced_accuracy.png", dpi=180, bbox_inches="tight"); plt.show()

# Primary thesis-safe strategy figure: best representative per strategy
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(strategy_mean["Strategy"], strategy_mean["Mean_macro_F1_present"])
ax.set_ylim(0, 1); ax.set_ylabel("Mean present-class Macro-F1")
ax.set_title("Learning-strategy ablation — best representative per strategy")
ax.grid(axis="y", alpha=0.3)
for i, v in enumerate(strategy_mean["Mean_macro_F1_present"]):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "ablation_learning_strategy.png", dpi=180, bbox_inches="tight")
plt.savefig(FIG_DIR / "ablation_learning_strategy_best_representative.png", dpi=180, bbox_inches="tight")
plt.show()

# Supplementary diagnostic: average over all tested configurations
if "strategy_mean_all_configs" in globals() and isinstance(strategy_mean_all_configs, pd.DataFrame):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(strategy_mean_all_configs["Strategy"], strategy_mean_all_configs["Mean_macro_F1_present"])
    ax.set_ylim(0, 1); ax.set_ylabel("Mean present-class Macro-F1")
    ax.set_title("Learning-strategy sensitivity — all configurations")
    ax.grid(axis="y", alpha=0.3)
    for i, v in enumerate(strategy_mean_all_configs["Mean_macro_F1_present"]):
        ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=8)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "ablation_learning_strategy_all_configurations.png", dpi=180, bbox_inches="tight")
    plt.show()


In [30]:
# 8 — Zip outputs for download
zip_path = shutil.make_archive(str(Path("/kaggle/working") / "N11_ablation_study_outputs"), "zip", root_dir=OUT_DIR)
print("✅ N11 ablation outputs saved:", zip_path)
print("Output files:")
for root, _, files in os.walk(OUT_DIR):
    for f in files:
        print(" -", Path(root, f).relative_to(OUT_DIR))


✅ N11 ablation outputs saved: /kaggle/working/N11_ablation_study_outputs.zip
Output files:
 - tables/n11_ablation_synthesis.csv
 - tables/ablation_3_learning_strategy_all_configurations.csv
 - tables/ablation_4_modal_density_sensitivity.csv
 - tables/ablation_1b_fedavg_vs_fedprox_matched_architecture.csv
 - tables/ablation_5_xgboost_class_weighting_on_off.csv
 - tables/ablation_1a_fedavg_vs_fedprox_best_family.csv
 - tables/ablation_2_architecture_comparison.csv
 - tables/best_model_per_client_before_ablation.csv
 - tables/ablation_1_fedavg_vs_fedprox.csv
 - tables/n11_ablation_synthesis_fixed.csv
 - tables/ablation_5_class_weighting_interpretation.csv
 - tables/n11_model_selection_table.csv
 - tables/best_model_per_strategy_before_ablation.csv
 - tables/ablation_3_learning_strategy_best_representative.csv
 - tables/ablation_3_learning_strategy.csv
 - figures/ablation_learning_strategy_best_representative.png
 - figures/ablation_learning_strategy.png
 - figures/ablation_class_weighting